### How FastAPI resolves dependencies on a route function

```py
@router.post(
    "/{movie_id}/video",
    response_model=VideoUploadResponseSchema,
    summary="Upload a video for a movie",
    status_code=status.HTTP_202_ACCEPTED,
)
async def upload_movie_video(
    movie_id: int,
    file: UploadFile,
    service: Annotated[VideoService, Depends(get_video_service)],
    _: Annotated[
        User,
        Depends(require_role(UserGroupEnum.MODERATOR, UserGroupEnum.ADMIN)),
    ],
    current_user: Annotated[User, Depends(get_current_user)],
) -> VideoUploadResponseSchema:
    """Upload a raw video file and queue HLS transcoding."""
    return await service.upload_video(movie_id, file, current_user.id)
```

When a request hits `POST /{movie_id}/video`, FastAPI inspects every parameter of `upload_movie_video` and classifies each one:

- **No annotation / path variable** → extracted from the URL path (`movie_id: int`)
- **`UploadFile`** → extracted from the multipart request body (`file: UploadFile`)
- **`Annotated[T, Depends(fn)]`** → handed off to FastAPI's dependency resolver

For `Depends` parameters, FastAPI builds a **dependency graph** — a tree where each node is a dependency function, and edges are its own `Depends` parameters. It resolves leaves first, then works inward, and each node is called at most once per request (results are cached and reused if the same dependency appears multiple times in the tree).

#### The dependency tree for `upload_movie_video`

```shell
upload_movie_video
├── get_video_service
│   ├── get_db              → AsyncSession
│   ├── get_s3_storage_client
│   │   └── get_settings    → BaseAppSettings
│   └── get_settings        → BaseAppSettings  (cached, called once)
│
├── require_role(MODERATOR, ADMIN)   ← returns role_checker
│   └── role_checker
│       └── get_current_user
│           ├── bearer_scheme       → HTTPAuthorizationCredentials
│           ├── get_jwt_auth_manager
│           │   └── get_settings    → BaseAppSettings  (cached again)
│           └── get_db              → AsyncSession  (cached again)
│
└── get_current_user         ← same function, FastAPI reuses cached result
```

FastAPI deduplicates: `get_settings`, `get_db`, `get_current_user` each appear multiple times in the tree but are called **exactly once** and their result is shared.

#### What `require_role(MODERATOR, ADMIN)` actually is

`require_role` is a **closure factory** — a plain `def` that returns an `async def`:

```python
def require_role(*allowed_roles):       # called at import time when decorating the route
    async def role_checker(             # this is what Depends() receives
        current_user: Annotated[User, Depends(get_current_user)],
    ) -> User:
        if not any(current_user.has_group(role) for role in allowed_roles):
            raise HTTPException(403, ...)
        return current_user
    return role_checker
```

When you write `Depends(require_role(MODERATOR, ADMIN))`:

1. `require_role(MODERATOR, ADMIN)` is evaluated **immediately** when Python loads the module — it creates a `role_checker` closure with `allowed_roles` baked in
2. That `role_checker` function is what `Depends()` receives — FastAPI treats it exactly like any other dependency function
3. On each request, FastAPI calls `role_checker`, which itself declares `Depends(get_current_user)` — so `get_current_user` is resolved first, then `role_checker` receives the already-resolved `User` object

#### Why `_` for the parameter name

```python
_: Annotated[
    User,
    Depends(require_role(UserGroupEnum.MODERATOR, UserGroupEnum.ADMIN)),
],
current_user: Annotated[User, Depends(get_current_user)],
```

FastAPI's dependency resolution is driven entirely by the **type annotation and `Depends()`**, not the parameter name. The name `_` is a Python convention for "I don't need this value in the function body" — it signals to readers that this dependency is declared purely for its **side effect** (the 403 check), not because the return value is used.

`current_user` is separately declared because the route body actually uses `current_user.id` — and since `get_current_user` is already in the tree via `role_checker`, FastAPI reuses the cached result at zero extra cost.

You could name `_` anything — `enforced_role`, `auth_guard` — `_` is just idiomatic Python for "intentionally unused".

---

### Why some dependencies `yield` and some just `return`
**The one-line explanation:** `Depends` doesn't just call a function and hand you the result — if that function is a generator, FastAPI treats it as a context manager and runs the code _after_ `yield` as guaranteed teardown, even if the route raised; if it's a plain function, there's no teardown phase at all, so you just `return`.

**Why it matters in interviews:** This is the single most common "explain `Depends` to me" interview question, and most candidates can use `get_db` but can't explain _why_ it's shaped that way. It also tests whether you understand `contextlib.AsyncExitStack`, which is the actual machinery underneath.

In [2]:
from collections.abc import AsyncGenerator
from contextlib import asynccontextmanager

from fastapi import FastAPI, Depends
from sqlalchemy import select
from sqlalchemy.ext.asyncio import AsyncSession, async_sessionmaker, create_async_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column


class Base(DeclarativeBase):
    pass


class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]


engine = create_async_engine("sqlite+aiosqlite:///./test.db")
AsyncSessionLocal = async_sessionmaker(engine, expire_on_commit=False)

In [3]:
# --- Pattern 1: bare async generator, used ONLY via Depends ---
async def get_db() -> AsyncGenerator[AsyncSession, None]:
    async with AsyncSessionLocal() as session:
        try:
            yield session          # <-- handed to the route
            await session.commit() # runs after the route returns normally
        except Exception:
            await session.rollback()
            raise                  # always re-raise, don't swallow

In [4]:
# --- Pattern 2: same logic, wrapped so it ALSO works outside Depends ---
get_db_contextmanager = asynccontextmanager(get_db)
# usage outside FastAPI: `async with get_db_contextmanager() as session: ...`

In [6]:
# --- Pattern 3: no resource to release, so no yield needed ---
class CacheService:
    def __init__(self, client):
        self.client = client


redis_client = object()  # stand-in


async def get_cache() -> CacheService:
    return CacheService(redis_client)


app = FastAPI()


@app.get("/users/{user_id}")
async def get_user(user_id: int, db: AsyncSession = Depends(get_db)):
    result = await db.execute(select(User).where(User.id == user_id))
    return result.scalar_one_or_none()

**What actually happens per request (the mechanics of `Depends`):**

1. FastAPI inspects the route's signature, finds every `Depends(...)` (including ones nested inside other dependencies — they form a tree, resolved depth-first).
2. **For each dependency**, it checks:
    - is this an `async def`? Awaited directly on the event loop
    - Is it a plain `def`? Run in a threadpool so it can't block the loop
    - Is it a generator (sync or async, `def`/`async def` with `yield`)? It gets entered into a per-request `AsyncExitStack`.
3. Results are cached per request by default (`use_cache=True`) — if two routes/sub-dependencies both depend on `get_db`, they get the _same_ session instance for that request, not two.
4. The route handler runs with the _yielded_ value injected.
5. When the route returns (or raises), FastAPI unwinds the `AsyncExitStack`, which resumes each generator past its `yield` — that's where `commit()`/`rollback()`/`close()` happens. If the route raised, the exception is thrown into the generator at the `yield` point via `.athrow()`, so your `try/except` around `yield` actually sees it.

---

**Expected output:** No exception → `INSERT/SELECT` succeeds, then `await session.commit()` runs after your handler's `return`. Exception in the handler → `await session.rollback()` runs, then the original exception propagates to FastAPI's exception handling — your data never gets half-committed.

**⚠️ INTERVIEW TRAP:** `async with AsyncSessionLocal() as session: yield session` _without_ the explicit `try/except/commit` looks correct and "works" in demos, but it never commits anything — `AsyncSession.__aexit__` only closes/rolls back, it doesn't commit on success. People ship this, autocommit-expecting behavior never arrives, and they "fix" it by sprinkling `await db.commit()` inside every route instead of centralizing it in the dependency.

**The `CacheService` case, contrasted:** `get_cache` has no per-request resource to acquire or release — the Redis client connection pool is already long-lived and shared. There's nothing to do "after the route is done," so there's no `yield`, no context manager, just call-and-return.

> The rule: **`yield` when the dependency owns something that must be released after the request, with release timing that matters (must happen post-response, must happen even on exception). Plain `return` when it's just construction/lookup.**

---

### Why using async `get_cache`
```py
async def get_cache() -> CacheService:
    return CacheService(redis_client)
```
It works (FastAPI accepts both sync and async dependency callables), but it's not the better choice here — and the reason matters for performance, given this project's load-testing focus.

**The mechanism:** FastAPI's dependency resolver checks whether each `Depends()` callable is a **coroutine function**. If it's `async def`, FastAPI `await`s it directly in the event loop. If it's a plain `def` (like your current `get_cache`), FastAPI instead dispatches it via `run_in_threadpool(call, ...)` — it hands the call off to a worker thread in Starlette's threadpool, waits for the result, then resumes. This exists so blocking sync code (e.g. a sync DB driver, file I/O) doesn't stall the event loop.

**The problem:** `get_cache()` does neither I/O nor blocking work — it just constructs `CacheService(redis_client)` and returns it. Routing it through the threadpool anyway means every single request that depends on it (likely most cached movie endpoints) consumes one of a limited number of threadpool slots for zero benefit, purely overhead. Under your documented 500-concurrent-user Locust scenario, that's needless contention on a shared, finite resource.

**Fix:** make it `async def` — no `await` needed inside, the async-ness alone is what tells FastAPI to call it inline instead of threadpool-dispatching it

---

**When NO explicit service's `await self.repo.commit()` used - ONLY `get_db` commits**
```py
async def like_movie(
    request: Request,
    movie_id: int,
    service: Annotated[
        MovieReactionService, Depends(get_movie_reaction_service)
    ],
    user: Annotated[User, Depends(get_current_user)],
) -> MovieReactionActionResponseSchema:
    """Like a movie."""
    return await service.like_movie(user_id=user.id, movie_id=movie_id)
```
The commit happened **after** `like_movie` returned its response object, but **before** the HTTP response was actually sent back to Swagger. Here's the exact sequence:

1. Router body runs: `service.like_movie(...)` executes `repo.upsert_movie_reaction(...)` — this just sends the `INSERT` to Postgres via the session (queued/executed on the connection but not yet committed), builds the response schema, and returns it.
2. `like_movie`'s `return` statement completes — the endpoint coroutine is now finished, FastAPI has the `MovieReactionActionResponseSchema` object **in hand**.
3. **Dependency teardown phase** — FastAPI/Starlette manages every `Depends()` that's an async generator (like `get_db`) using an `AsyncExitStack` for the request. At this point it unwinds that stack, which means: resume `get_db` exactly where it left off, right after `yield session`. That's where `await session.commit()` actually executes — _outside_ `like_movie`'s function body entirely, in framework-owned cleanup code you never call directly.
4. Only once that `commit()` finishes successfully does FastAPI serialize the response schema to JSON and transmit the `200 OK` back to Swagger.

So the ordering is: **route body finishes → `get_db` teardown commits → response is sent.** This also explains an important edge case: if the commit itself failed (e.g. a constraint violation that only surfaces at flush/commit time), the client would get a `500`, not the `200` the route "thought" it was returning — because the response hasn't been sent yet when that commit runs.

---

### How many times does `AsyncSessionLocal()` actually get called for one request
If `get_db` is depended on by both a route's top-level parameter **and** by another dependency that route also uses (e.g., a `get_current_user` dependency that itself does `db: AsyncSession = Depends(get_db)`), the `AsyncSessionLocal()` is called **once**, since results are cached per request by default (`use_cache=True`) — if two routes/sub-dependencies both depend on `get_db`, they get the **same session instance** for that request, not two.

**Why:** `Depends` caches by the dependency _callable_ (the `get_db` function object itself) within the scope of one request. The first time anything needs `get_db`, FastAPI enters its generator, gets the session, and stores the result in that request's dependency cache keyed on `get_db`. Every other place in the same dependency tree that also declares `Depends(get_db)` — whether it's the route directly or nested three levels deep inside `get_current_user` — just receives the cached session object back. No second `async with AsyncSessionLocal()`, no second connection checkout.

This is also exactly why the commit/rollback belongs in `get_db` itself rather than scattered in routes: there's only ever one session and one teardown point for the whole request, so centralizing the transaction boundary there is correct, not just convenient.

**⚠️ INTERVIEW TRAP corollary:** if you ever pass `use_cache=False` to one of the `Depends(...)` calls for the same callable, you _break_ this guarantee — you'd get two independent sessions in one request, each with its own transaction, and now `get_current_user`'s query and your route's query can't see each other's uncommitted writes. This is a deliberate escape hatch (rare, e.g. wanting a fresh independent instance), not a default you'd reach for with DB sessions.

---

### Two ways declare dependencies
```py
@router.post(
    "",
    dependencies=[
        Depends(require_role(UserGroupEnum.MODERATOR, UserGroupEnum.ADMIN))
    ],
    ...
)
async def create_movie(
    data: MovieCreateSchema,
    service: Annotated[MovieService, Depends(get_movie_service)],
) -> MovieDetailSchema:
    """Create a new movie. Requires MODERATOR or ADMIN role."""
    return await service.create_movie(data)
```
FastAPI has two ways to declare a dependency:

1. **Function parameter** — `user: Annotated[User, Depends(get_current_user)]` — use this when you need the return value inside the handler (e.g. `user.id`).
2. **`dependencies=[]` on the decorator** — use this when you only care about the side effect (the 403 check), not the return value. The dependency still runs and can raise `HTTPException`, but the handler receives nothing from it.

`create_movie` doesn't need to know _who_ the user is — it just needs to **block non-admins**. So `require_role` sits in `dependencies=[...]` and the handler signature stays clean.

In `upload_video`, the endpoint needs `user_id` to record the uploader, so `require_role` appears as a `_` parameter to capture the user object — that's the other pattern you saw.
```py
@router.post(
    "/{movie_id}/video",
    response_model=VideoUploadResponseSchema,
    summary="Upload a video for a movie",
    ...
    status_code=status.HTTP_202_ACCEPTED,
)
async def upload_movie_video(
    movie_id: int,
    file: UploadFile,
    service: Annotated[VideoService, Depends(get_video_service)],
    _: Annotated[
        User,
        Depends(require_role(UserGroupEnum.MODERATOR, UserGroupEnum.ADMIN)),
    ],
    current_user: Annotated[User, Depends(get_current_user)],
) -> VideoUploadResponseSchema:
    """Upload a raw video file and queue HLS transcoding."""
    return await service.upload_video(movie_id, file, current_user.id)
```
